# Day 20: Transfer Learning & Fine-Tuning

> **Goal:** Use a pre-trained CNN (ResNet-18 trained on ImageNet) and fine-tune it for a new task with minimal data.

---

## Why Transfer Learning?

Training a large CNN from scratch requires:
- Millions of labelled images
- Days of GPU time
- Expert tuning

**Transfer learning** takes a model pre-trained on a huge dataset (ImageNet: 1.2M images, 1000 classes) and adapts it to your task. The early-layer features (edges, textures) are universal!

### Two strategies
| Strategy | What changes | When to use |
|---|---|---|
| **Feature extraction** | Only the final classifier layer | Very small dataset (<1000 images) |
| **Fine-tuning** | Some or all layers, lower LR | Medium dataset, similar domain |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Load Pre-trained ResNet-18

In [ ]:
# Load ResNet-18 with ImageNet weights
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("ResNet-18 architecture:")
print(resnet18)

# Check the final layer
print(f"\nOriginal FC layer: {resnet18.fc}")
print(f"Output classes: 1000 (ImageNet)")

---
## 2. Prepare CIFAR-10 with ImageNet-style Transforms

ImageNet models expect:
- Input size: 224×224
- Normalization with ImageNet mean/std

In [ ]:
# ImageNet normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# CIFAR-10 is 32×32, but ResNet expects 224×224
# We'll resize (or adjust the model — see below)
train_transform = transforms.Compose([
    transforms.Resize(224),                        # Resize to 224×224
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_data, val_data = random_split(full_train, [45000, 5000],
                                     generator=torch.Generator().manual_seed(42))

# Smaller batch size due to larger images
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

classes = full_train.classes
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

---
## 3. Strategy A: Feature Extraction (Freeze everything)

1. Replace the final FC layer (1000 → 10 classes)
2. Freeze all other layers
3. Only train the new FC layer

In [ ]:
def create_feature_extractor():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze ALL parameters
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final FC layer (the only trainable part)
    num_features = model.fc.in_features  # 512
    model.fc = nn.Linear(num_features, 10)

    return model

model_feat = create_feature_extractor().to(device)

# Check: how many trainable params?
total = sum(p.numel() for p in model_feat.parameters())
trainable = sum(p.numel() for p in model_feat.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,} ({trainable/total:.2%})")

Notice how few parameters are trainable — we're only learning 5,130 weights (512×10 + 10 bias) out of 11 million total! The frozen backbone acts as a fixed feature extractor, converting images into 512-dimensional representations that the new FC layer learns to classify.

Now let's train and see how well ImageNet features transfer to CIFAR-10:

In [ ]:
# Training helpers
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
# Train feature extractor (only FC layer)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_feat.fc.parameters(), lr=1e-3)  # Only FC params!

print("Feature Extraction (frozen backbone):")
for epoch in range(5):
    t0 = time.time()
    tl, ta = train_one_epoch(model_feat, train_loader, loss_fn, optimizer, device)
    vl, va = evaluate(model_feat, val_loader, loss_fn, device)
    print(f"  Epoch {epoch+1}: Train {ta:.2%}, Val {va:.2%} ({time.time()-t0:.1f}s)")

---
## 4. Strategy B: Fine-Tuning (Unfreeze some/all layers)

Start from the feature extractor, then unfreeze and train with a **lower learning rate**.

In [ ]:
def create_finetuned_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    # Replace FC layer
    model.fc = nn.Linear(model.fc.in_features, 10)
    # All parameters are trainable (no freezing)
    return model

model_ft = create_finetuned_model().to(device)

# Use different learning rates: lower for backbone, higher for new FC
optimizer_ft = optim.Adam([
    {"params": [p for n, p in model_ft.named_parameters() if "fc" not in n], "lr": 1e-4},
    {"params": model_ft.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=10)

trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (ALL)")

**Key technique: Differential learning rates.** The backbone (pre-trained layers) gets a learning rate 10× smaller than the new FC layer. This preserves the useful pre-trained features while allowing the model to adapt to our task. Without this, aggressive updates could destroy what the model already learned — a phenomenon called **catastrophic forgetting**.

In [ ]:
# Train with fine-tuning
print("\nFine-tuning (all layers):")
history = {"val_acc": []}

for epoch in range(10):
    t0 = time.time()
    tl, ta = train_one_epoch(model_ft, train_loader, loss_fn, optimizer_ft, device)
    vl, va = evaluate(model_ft, val_loader, loss_fn, device)
    scheduler.step()
    history["val_acc"].append(va)
    if (epoch+1) % 2 == 0:
        print(f"  Epoch {epoch+1}: Train {ta:.2%}, Val {va:.2%} ({time.time()-t0:.1f}s)")

test_loss, test_acc = evaluate(model_ft, test_loader, loss_fn, device)
print(f"\n🎯 Test Accuracy: {test_acc:.2%}")

---
## 5. Summary: Transfer Learning Workflow

```python
# Step 1: Load pre-trained model
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Step 2: Replace classifier for your task
model.fc = nn.Linear(512, num_your_classes)

# Step 3: Choose strategy
#   A) Freeze backbone → train only FC
#   B) Fine-tune all → use lower LR for backbone

# Step 4: Train!
```

---
## 🧪 Exercises

### Exercise 1: Gradual unfreezing
Freeze everything, train FC for 3 epochs. Then unfreeze layer4 and train for 3 more. Then unfreeze layer3 and train 3 more. Plot the progression.

### Exercise 2: Different backbones
Try `models.resnet34`, `models.mobilenet_v2`, or `models.efficientnet_b0`. Compare accuracy and training speed.

### Exercise 3: Small data regime
Use only 500 CIFAR-10 training samples. Compare: (a) CNN from scratch, (b) feature extraction, (c) fine-tuning. Which wins?

### Exercise 4: Feature similarity
Extract features from the pre-trained model (before fine-tuning). How well do they cluster by CIFAR-10 class?

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Transfer learning** | Use pre-trained features instead of training from scratch |
| **Feature extraction** | Freeze backbone, train only classifier |
| **Fine-tuning** | Train all layers with lower LR for backbone |
| **Differential LR** | Backbone gets 10× lower LR than new layers |
| **ImageNet features** | Universal edges/textures transfer to most vision tasks |

**Tomorrow:** We start building a CNN from scratch on a custom dataset — putting everything together!